parent: Preselection_v6

In [1]:
import os, sys
import numpy as np
import pickle
import uproot
import glob
import pandas as pd
from tqdm import tqdm
import json
import ROOT as rt
import fnmatch

## user define
sys.path.append('.')
sys.path.append('./Utils')
from Plotter import HistStack

# from beeps import *

Welcome to JupyROOT 6.28/06


# Set config

In [2]:
# ERA="ee17"
# ERA="ee16pre"
# # ERA="mm18"
# ERA="em18"
# # DATE = '20230129_17_eleffmva'
# DATE = '20230301_18'
# # DATE = '20230130_18_dyhighworctight'
# # INPUTDIR = "/eos/user/h/hkwon/Run2/20230130_18_dyhighworctight/mm/"
# INPUTDIR = "/eos/user/h/hkwon/Run2/20230301_18/em/"
import sys
ERA = sys.argv[1]
DATE = sys.argv[2]
INPUTDIR = sys.argv[3]
REGION = sys.argv[4]

DUMP_PKL = False
DUMP_SIDEBAND_MET_PKL = False
DUMP_SIDEBAND_MASS_PKL = False
DUMP_SIDEBAND_PT_PKL = False
# DUMP_DIVIDED_BY_BTAG = True
# DRAW_SR = True
# RUN_SIDEBAND_MET = False
# RUN_SIDEBAND_MASS = False
# RUN_SIDEBAND_PT = False
# RUN_BTAG = True
# EXPORT_VARIABLE = False

# for dev
# ERA = "mm16pre"

year = ERA.replace("em", "")
year = year.replace("mm", "")
year = year.replace("ee", "")
channel = ERA.replace("16pre", "")
channel = channel.replace("16post", "")
channel = channel.replace("17", "")
channel = channel.replace("18", "")

# DATE = "20231214GEv1"
# INPUTDIR = "/eos/user/h/hkwon/Run2/"+DATE+"_"+year+"/"+channel+"/"


# Get input

In [3]:
from warnings import simplefilter 
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

lumi_all={
    "ee16pre":19520,
    "ee16post":16810,
    "ee17":41480,
    "ee18":59830,
    "mm16pre":19520,
    "mm16post":16810,
    "mm17":41480,
    "mm18":59830,    
#     "mm18":14000,    
    "em16pre":19520,
    "em16post":16810,
    "em17":41480,
    "em18":59830     
}

data_evts = {
    "ee16pre":652566543,
    "ee16post":287977113,
    "ee17":460041183,
    "ee18":1326465857,
    "mm16pre":458655573,
    "mm16post":327037312,
    "mm17":739073837,
#     "mm18":944842550,  #949352777 
    "mm18":949352777,  #949352777 
    "em16pre":1111222116,
    "em16post":615014425,
    "em17":1199115020,
#     "em18":2271308407   #2281001366  
    "em18":2281001366   #2281001366  
}

lumi=lumi_all[ERA]
print(ERA, lumi)


def get_df(root_file_name, unneccesary_columns):
    f = uproot.open(root_file_name)
    df = uproot.open(root_file_name)["Events_"+channel].arrays(library="pd")
    df_met_u = uproot.open(root_file_name)["Events_"+channel+"_met_u"].arrays(library="pd")
    df_met_d = uproot.open(root_file_name)["Events_"+channel+"_met_d"].arrays(library="pd")     
    df_met_jer_u = uproot.open(root_file_name)["Events_"+channel+"_met_jer_u"].arrays(library="pd")
    df_met_jer_d = uproot.open(root_file_name)["Events_"+channel+"_met_jer_d"].arrays(library="pd")    
    df_met_ue_u = uproot.open(root_file_name)["Events_"+channel+"_met_ue_u"].arrays(library="pd")
    df_met_ue_d = uproot.open(root_file_name)["Events_"+channel+"_met_ue_d"].arrays(library="pd") 
    df_scale_u = uproot.open(root_file_name)["Events_"+channel+"_scale_u"].arrays(library="pd")
    df_scale_d = uproot.open(root_file_name)["Events_"+channel+"_scale_d"].arrays(library="pd")
    info = uproot.open(root_file_name)["Info_"+channel].arrays(library="pd")
#     return sum(info["sumweight"]), df.drop(unneccesary_columns, axis=1), df_met_u.drop(unneccesary_columns, axis=1), df_met_d.drop(unneccesary_columns, axis=1), df_scale_u.drop(unneccesary_columns, axis=1), df_scale_d.drop(unneccesary_columns, axis=1)
    return sum(info["sumweight"]), df.drop(unneccesary_columns, axis=1), df_met_u.drop(unneccesary_columns, axis=1), df_met_d.drop(unneccesary_columns, axis=1), df_met_jer_u.drop(unneccesary_columns, axis=1), df_met_jer_d.drop(unneccesary_columns, axis=1), df_met_ue_u.drop(unneccesary_columns, axis=1), df_met_ue_d.drop(unneccesary_columns, axis=1), df_scale_u.drop(unneccesary_columns, axis=1), df_scale_d.drop(unneccesary_columns, axis=1)

unneccesary_columns = []
dic_input = {
    "ee16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/ee/",
    "mm16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/mm/",
    "em16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/em/",
    "ee16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/ee/",
    "mm16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/mm/",
    "em16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/em/",
    "ee17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/ee/",
    "mm17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/mm/",
    "em17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/em/",
    "ee18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/ee/",
    "mm18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/mm/",    
    "em18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/em/"        
#     "em18": "/eos/user/h/hkwon/Run2/20221120_18/em/"        
}
# input_dir = dic_input[ERA]
input_dir = INPUTDIR
f = []
for (dirpath, dirnames, filenames) in os.walk(input_dir):
    f.extend(filenames)
    break

dic_tup={}
for file in tqdm(f):
    # file naming: flatNtuple_Preselect_WZ1.root, flatNtuple_Preselect_Zp-3700_CH-1345.root
    if not "flatNtuple" in file:
        continue
#     if channel=="mm" and (fnmatch.fnmatch(file, '*E*Run*') or fnmatch.fnmatch(file, '*M*RunB*') or fnmatch.fnmatch(file, '*M*RunC*') or fnmatch.fnmatch(file, '*M*RunD*')): #temp
#     if channel=="mm" and fnmatch.fnmatch(file, '*E*Run*'): #temp
#         continue
#     if channel=="ee" and fnmatch.fnmatch(file, '*M*Run*'):
#         continue
#     if channel=="em" and fnmatch.fnmatch(file, '*E*Run*'):
#         continue 
    if "Run" in file and not "hadd" in file:
        continue
    if "TT" in file and not "hadd" in file:
        continue
    if "DY" in file and not "hadd" in file and not "To" in file:
        continue  
    if "TW" in file and not "hadd" in file:
        continue         
    if "WW" in file and not "hadd" in file:
        continue  
    if "WZ" in file and not "hadd" in file:
        continue      
    if "ZZ" in file and not "hadd" in file:
        continue      
    if "Htt" in file and not "hadd" in file:
        continue  
        
    if "Zp" in file:
        sample=file.split('_')[2]+"_"+file.split('_')[3]
    else:
        print(file)
        sample=file.split('_')[2]
    sample=sample.rstrip('.root')
    try:
        print(sample)
        dic_tup[sample]=get_df(input_dir+file, unneccesary_columns)   
    except:
        print(sample)
        # beep()        
        if "Run" in sample: # check file corrupt
            raise    
# dic_tup

mm16pre 19520



  0%|          | 0/182 [00:00<?, ?it/s]

flatNtuple_Preselect_DY1000To15000.root
DY1000To15000
DY1000To15000
flatNtuple_Preselect_DY1000To15001.root
DY1000To15001



  2%|▏         | 4/182 [00:01<01:16,  2.31it/s]

flatNtuple_Preselect_DY1000To15002.root
DY1000To15002


...


 99%|█████████▉| 180/182 [02:08<00:03,  1.77s/it]

flatNtuple_Preselect_antitop_hadd.root
antitop



 99%|█████████▉| 181/182 [02:10<00:01,  1.85s/it]

flatNtuple_Preselect_top_hadd.root
top


100%|██████████| 182/182 [02:12<00:00,  1.38it/s]


### check missing signals

In [5]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
ALL_MASSES = setting["ALL_MASSES"]

if REGION == "SR":
    for m in ALL_MASSES:
        find = False
        for key in dic_tup:
            if "Zp" in key and key==m:
                find = True
        if find!=True:
            print("WARNING: missing", m)
            sys.exit()
                

SystemExit: 

/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3449: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
# beep_ok()

In [5]:
# for key in dic_tup:
#     print(key)

In [6]:
# get x-sec
dic_xsec={}
with open('config/CrossSection.json') as json_file:
    data = json.load(json_file)
    dic_xsec=data
print(dic_xsec)

{'Data': 1, 'DY': 6077.22, 'DY500To700': 0.2558, 'DY700To800': 0.04023, 'DY800To1000': 0.03406, 'DY1000To1500': 0.01828, 'DY1500To2000': 0.002367, 'DY2000To3000': 0.0005409, 'TT': 88.29, 'TTUp': 88.29, 'TTDown': 88.29, 'TThdampUp': 88.29, 'TThdampDown': 88.29, 'TTSemi': 365.34, 'TTZ': 0.2432, 'top': 34.91, 'antitop': 34.91, 'WW': 12.82, 'WZ': 5.23, 'WZQQ': 6.42, 'ZZ': 0.6008, 'WWW': 0.0072, 'WWZ': 0.1707, 'HWW': 0.31, 'H': 2.78, 'Zp-1700_CH-345': 0.03523, 'Zp-1700_CH-595': 0.03523, 'Zp-1700_CH-845': 0.03523, 'Zp-2100_CH-345': 0.01198, 'Zp-2100_CH-595': 0.01198, 'Zp-2100_CH-845': 0.01198, 'Zp-2500_CH-345': 0.004396, 'Zp-2500_CH-595': 0.004396, 'Zp-2500_CH-845': 0.004396, 'Zp-2500_CH-1095': 0.004396, 'Zp-2900_CH-345': 0.001768, 'Zp-2900_CH-595': 0.001768, 'Zp-2900_CH-845': 0.001768, 'Zp-2900_CH-1095': 0.001768, 'Zp-2900_CH-1345': 0.001768, 'Zp-3300_CH-345': 0.000729, 'Zp-3300_CH-595': 0.000729, 'Zp-3300_CH-845': 0.000729, 'Zp-3300_CH-1095': 0.000729, 'Zp-3300_CH-1345': 0.000729, 'Zp-3300

In [7]:
merged_dic_df={}
for key in dic_tup:       
    merged_dic_df[key]=dic_tup[key]

In [8]:
# # free memory 
# dic_tup.clear()

In [9]:
totalinput = """
######################################################
####   Total input number of events (sumweight)   ####
######################################################
"""


# merged_dic_df["Zp-2500"]=dic_tup["Zp-2500"][0], dic_tup["Zp-2500"][1] 
hadded = ["Data", "TT", "TTSemi", "TTUp", "TTDown", "TThdampUp", "TThdampDown", "top", "antitop", "WW", "WZ", "WZQQ", "DY", "TTZ", "ZZ", "H", "HWW", "WWW", "WWZ"]
for p in hadded:
    line = p + ":" + str(merged_dic_df[p][0]) + '\n'
    totalinput += line
    
print(totalinput)


######################################################
####   Total input number of events (sumweight)   ####
######################################################
Data:739073837
TT:105859990
TTSemi:343257666
TTUp:40900188
TTDown:38977214
TThdampUp:38588816
TThdampDown:38744898
top:5447726
antitop:5673700
WW:7071358
WZ:6275882
WZQQ:18136498
DY:131545008
TTZ:6653168
ZZ:35643944
H:3701740
HWW:6440491
WWW:3477806
WWZ:8314612



In [10]:
out_dir = DATE+"/Presel/DUMP/"+ERA

In [11]:
os.system("mkdir -p "+out_dir)
with open(out_dir+'/Nevents.txt', 'w') as f:
    f.write(totalinput)

In [12]:
print("expected:", data_evts[ERA], ", while: ", merged_dic_df['Data'][0])

try:
   if data_evts[ERA] > 1.01*merged_dic_df['Data'][0] or data_evts[ERA] < 0.99*merged_dic_df['Data'][0]: # check if more than 1% differ
       raise ValueError('Data events missing!')
except (ValueError, IndexError):
#    beep()
   print("WARNING! expected:", data_evts[ERA], ", while: ", merged_dic_df['Data'][0]) 
   raise

expected: 739073837 , while:  739073837


### NaN cleaner

In [13]:
# temp!!
for key in merged_dic_df:
    df = merged_dic_df[key][1]
    df_met_u = merged_dic_df[key][2]
    df_met_d = merged_dic_df[key][3]  
    df_met_jer_u = merged_dic_df[key][4]
    df_met_jer_d = merged_dic_df[key][5]
    df_met_ue_u = merged_dic_df[key][6]
    df_met_ue_d = merged_dic_df[key][7]
    df_scale_u = merged_dic_df[key][8]
    df_scale_d = merged_dic_df[key][9]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(axis=1)].index
        print(i)
        df_new = df.drop(i)
        df_met_u_new = df_met_u.drop(i)
        df_met_d_new = df_met_d.drop(i)            
        df_met_jer_u_new = df_met_jer_u.drop(i)
        df_met_jer_d_new = df_met_jer_d.drop(i)        
        df_met_ue_u_new = df_met_ue_u
        df_met_ue_d_new = df_met_ue_d        
        df_scale_u_new = df_scale_u
        df_scale_d_new = df_scale_d
#         print(df_new.isnull().values.any())
        #workaround for tuple assignment
#         merged_dic_df[key] = merged_dic_df[key][0], df_new, df_met_u_new, df_met_d_new, df_scale_u, df_scale_d
        merged_dic_df[key] = merged_dic_df[key][0], df_new, df_met_u_new, df_met_d_new, df_met_jer_u_new, df_met_jer_d_new, df_met_ue_u_new, df_met_ue_d_new, df_scale_u_new, df_scale_d_new
        


Warning, nan in DY2000To30000
Int64Index([360, 1414, 3488], dtype='int64')
Warning, nan in WWW
Int64Index([14730, 22095, 31537], dtype='int64')
Warning, nan in WWZ
Int64Index([4389, 9413], dtype='int64')
Warning, nan in WW
Int64Index([11902], dtype='int64')
Warning, nan in Zp-3700_CH-845
Int64Index([    0,     1,     2,     3,     4,     5,     6,     7,     8,
                9,
            ...
            26517, 26518, 26519, 26520, 26521, 26522, 26523, 26524, 26525,
            26526],
           dtype='int64', length=26527)
Warning, nan in antitop
Int64Index([177], dtype='int64')


In [14]:
pd.options.mode.chained_assignment = None
# sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'topSF_pre', 'genweight_pre', 'trigSF_pre']
sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'effrecoSF_pre', 'topSF_pre', 'genweight_pre', 'trigSF_pre', 'btagSF_pre'] #, 'btagSF_cor_pre'
# sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'topSF_pre', 'genweight_pre']
eventweights = """
####################################################
####       event weights (normalization)        ####
####################################################
"""

def get_dic_with_weight(merged_dic_df):
    dic_df={}
    dic_df_met_u={}
    dic_df_met_d={}    
    dic_df_met_jer_u={}
    dic_df_met_jer_d={}
    dic_df_met_ue_u={}
    dic_df_met_ue_d={}
    dic_df_scale_u={}
    dic_df_scale_d={}
    for key, vs in merged_dic_df.items():
        print(key) 
        if vs[0]==0:
            print("missing: ", key, "!, continue without it")
            continue
        if "Data" in key:
            norm=1
        else:
            if fnmatch.fnmatch(key, '*DY*To*'):
                key = key[:len(key)-1]
            norm=lumi*dic_xsec[key]/vs[0]
        print(key, ":", norm)
        global eventweights
        eventweights+=key+":"+str(norm)+"\n"
#         print(eventweights)
        dic_df[key]=vs[1]
        dic_df_met_u[key] = vs[2]
        dic_df_met_d[key] = vs[3]        
        dic_df_met_jer_u[key] = vs[3]
        dic_df_met_jer_d[key] = vs[4]
        dic_df_met_ue_u[key] = vs[5]
        dic_df_met_ue_d[key] = vs[6]
        dic_df_scale_u[key] = vs[7]
        dic_df_scale_d[key]=vs[8]
        dic_df[key]["weight"] = vs[1]["genweight_pre"]*norm
#         dic_df[key]["lumifilter"] = np.ones(vs[1]["genweight_pre"].shape[0])
        a = np.ones(vs[1]["genweight_pre"].shape[0])
        a[int(vs[1]["genweight_pre"].shape[0]*1/3):] = 0
        if key=="Data":
            a = np.zeros(vs[1]["genweight_pre"].shape[0])
#         print(a)
        dic_df_met_u[key]["weightmetUp"] = vs[1]["genweight_pre"]*norm
        dic_df_met_d[key]["weightmetDown"] = vs[1]["genweight_pre"]*norm        
        dic_df_met_jer_u[key]["weightmetjerUp"] = vs[1]["genweight_pre"]*norm
        dic_df_met_jer_d[key]["weightmetjerDown"] = vs[1]["genweight_pre"]*norm
        dic_df_met_ue_u[key]["weightmetueUp"] = vs[1]["genweight_pre"]*norm
        dic_df_met_ue_d[key]["weightmetueDown"] = vs[1]["genweight_pre"]*norm
        dic_df_scale_u[key]["weightscaleUp"] = vs[1]["genweight_pre"]*norm
        dic_df_scale_d[key]["weightscaleDown"] = vs[1]["genweight_pre"]*norm
#         dic_df_met_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_d[key]["hemveto"] = vs[1]["hemveto"]       
#         dic_df_met_jer_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_jer_d[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_ue_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_ue_d[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_scale_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_scale_d[key]["hemveto"] = vs[1]["hemveto"]
        dic_df[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_d[key]["hemfilter"] = a + vs[1]["hemveto"]    
        dic_df_met_jer_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_jer_d[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_ue_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_ue_d[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_scale_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_scale_d[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df[key]["wopu"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["wol1"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["woeff"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["wotop"] = vs[1]["genweight_pre"]*norm       
        dic_df[key]["wobsf"] = vs[1]["genweight_pre"]*norm       
      
        for s in sys:
            if s=="genweight_pre" or s=="btagSF_cor_pre":
                continue
            dic_df[key]["weight"] *= vs[1][s]
            dic_df_met_u[key]["weightmetUp"] *= vs[1][s]
            dic_df_met_d[key]["weightmetDown"] *= vs[1][s]            
            dic_df_met_jer_u[key]["weightmetjerUp"] *= vs[1][s]
            dic_df_met_jer_d[key]["weightmetjerDown"] *= vs[1][s]            
            dic_df_met_ue_u[key]["weightmetueUp"] *= vs[1][s]
            dic_df_met_ue_d[key]["weightmetueDown"] *= vs[1][s]            
            dic_df_scale_u[key]["weightscaleUp"] *= vs[1][s]
            dic_df_scale_d[key]["weightscaleDown"] *= vs[1][s]       

            # n-1 correction
            if s!="puweight_pre":
                dic_df[key]["wopu"] *= vs[1][s]
            if s!="l1prefiring_pre":
                dic_df[key]["wol1"] *= vs[1][s]
            if s!="effSF_pre":
                dic_df[key]["woeff"] *= vs[1][s]
#             if s!="topSF_pre":
            if not "top" in s:
                dic_df[key]["wotop"] *= vs[1][s]
            if s!="btagSF_pre":
                dic_df[key]["wobsf"] *= vs[1][s]

        dic_df[key]["weighthdampUp"] = dic_df[key]["weight"].copy(deep=True)
        dic_df[key]["weighthdampDown"] = dic_df[key]["weight"].copy(deep=True)
        dic_df[key]["weightPSUp"] = dic_df[key]["weight"].copy(deep=True)
        dic_df[key]["weightPSDown"] = dic_df[key]["weight"].copy(deep=True)            
        dic_df[key]["weightJECUp"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jes_u_pre"]            
        dic_df[key]["weightJECDown"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jes_d_pre"]         
        dic_df[key]["weightJERUp"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jer_u_pre"]            
        dic_df[key]["weightJERDown"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jer_d_pre"]                      
            
        # up, down    
        for s1 in sys:
            #if "top" in s1 or "eff" in s1:
#             if "btag" in s1:
#                 continue
            if s1=="btagSF_pre":
                continue
            s_u = s1.replace("_pre", "_u_pre")
            s_d = s1.replace("_pre", "_d_pre")
            if s1=="genweight_pre":
                dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_u])
                dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_d])
#             elif s1=="btagSF_cor_pre":   
#                 dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_u]
#                 dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_d] 
            else:
                dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_u]
                dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_d]  
                
        # for b-tag
        btag_sys = ['btagSFbc_pre', 'btagSFbc_correlated_pre', 'btagSFlight_pre', 'btagSFlight_correlated_pre']
        for bsys in btag_sys:
            s_u = bsys.replace("_pre", "_u_pre")
            s_d = bsys.replace("_pre", "_d_pre")
            dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_u]
            dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_d]
        
                
#     return dic_df, dic_df_met_u, dic_df_met_d, dic_df_scale_u, dic_df_scale_d
    return dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_jer_u, dic_df_met_jer_d, dic_df_met_ue_u, dic_df_met_ue_d, dic_df_scale_u, dic_df_scale_d

# dic_df, dic_df_met_u, dic_df_met_d, dic_df_scale_u, dic_df_scale_d = get_dic_with_weight(merged_dic_df)
dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_jer_u, dic_df_met_jer_d, dic_df_met_ue_u, dic_df_met_ue_d, dic_df_scale_u, dic_df_scale_d = get_dic_with_weight(merged_dic_df)


DY1000To15000
DY1000To1500 : 0.006725273399735692
DY1500To20000
DY1500To2000 : 0.0008913829700309587
DY2000To30000
DY2000To3000 : 0.00019855513765608545
DY500To7000
DY500To700 : 0.0660359101064856
DY700To8000
DY700To800 : 0.014978640672124085
DY800To10000
DY800To1000 : 0.012300589428593817
DY
DY : 1.916325746089886
Data
Data : 1
HWW
HWW : 0.0019965558526516066
H
H : 0.03115140447465246
TTDown
TTDown : 0.09395923474674203
TTSemi
TTSemi : 0.0441484770801885
TTUp
TTUp : 0.08954162264486414
TTZ
TTZ : 0.0015162605243096222
TT
TT : 0.03459540474167814
TThdampDown
TThdampDown : 0.09452261817801147
TThdampUp
TThdampUp : 0.09490493825983155
WWW
WWW : 8.587483028092998e-05
WWZ
WWZ : 0.0008515894668326074
WW
WW : 0.0752010575620694
WZQQ
WZQQ : 0.014683187459894406
WZ
WZ : 0.03456731659390665
ZZ
ZZ : 0.0006991702152825737
Zp-1700_CH-345
Zp-1700_CH-345 : 0.009742269333333333
Zp-1700_CH-595
Zp-1700_CH-595 : 0.009742269333333333
Zp-1700_CH-845
Zp-1700_CH-845 : 0.009742269333333333
Zp-2100_CH-345
Zp-2

In [15]:
pd.set_option('display.max_columns', None)
dic_df["TT"][:10]

,PuppiMET_pre,mass_pre,pt_lead_pre,pt_trail_pre,U_pre,JZB_pre,dPhill_pre,dPhilMET_pre,MT_pre,MT2_pre,Zpt_pre,dRll_pre,dPhillMET_pre,eta_lead_pre,eta_trail_pre,njet_pre,nbjetflav_pre,nbjetflav_jesup_pre,nbjetflav_jesdown_pre,nbjetflav_jerup_pre,nbjetflav_jerdown_pre,HT_pre,PuppiMET_phi_pre,MET_pre,MET_phi_pre,phi_lead_pre,phi_trail_pre,hemveto,genweight_u_pre,genweight_d_pre,genweight_pre,puweight_pre,puweight_u_pre,puweight_d_pre,l1prefiring_pre,l1prefiring_u_pre,l1prefiring_d_pre,effrecoSF_pre,effrecoSF_u_pre,effrecoSF_d_pre,effSF_pre,effSF_u_pre,effSF_d_pre,trigSF_pre,trigSF_u_pre,trigSF_d_pre,topSF_pre,topSF_u_pre,topSF_d_pre,btagSF_pre,btagSFbc_u_pre,btagSFbc_d_pre,btagSFbc_correlated_u_pre,btagSFbc_correlated_d_pre,btagSFlight_u_pre,btagSFlight_d_pre,btagSFlight_correlated_u_pre,btagSFlight_correlated_d_pre,btagSF_jes_u_pre,btagSF_jes_d_pre,btagSF_jer_u_pre,btagSF_jer_d_pre,pdf0_pre,pdf1_pre,pdf2_pre,pdf3_pre,pdf4_pre,pdf5_pre,pdf6_pre,pdf7_pre,pdf8_pre,pdf9_pre,pdf10_pre,pdf11_pre,pdf12_pre,pdf13_pre,pdf14_pre,pdf15_pre,pdf16_pre,pdf17_pre,pdf18_pre,pdf19_pre,pdf20_pre,pdf21_pre,pdf22_pre,pdf23_pre,pdf24_pre,pdf25_pre,pdf26_pre,pdf27_pre,pdf28_pre,pdf29_pre,pdf30_pre,pdf31_pre,pdf32_pre,pdf33_pre,pdf34_pre,pdf35_pre,pdf36_pre,pdf37_pre,pdf38_pre,pdf39_pre,pdf40_pre,pdf41_pre,pdf42_pre,pdf43_pre,pdf44_pre,pdf45_pre,pdf46_pre,pdf47_pre,pdf48_pre,pdf49_pre,pdf50_pre,pdf51_pre,pdf52_pre,pdf53_pre,pdf54_pre,pdf55_pre,pdf56_pre,pdf57_pre,pdf58_pre,pdf59_pre,pdf60_pre,pdf61_pre,pdf62_pre,pdf63_pre,pdf64_pre,pdf65_pre,pdf66_pre,pdf67_pre,pdf68_pre,pdf69_pre,pdf70_pre,pdf71_pre,pdf72_pre,pdf73_pre,pdf74_pre,pdf75_pre,pdf76_pre,pdf77_pre,pdf78_pre,pdf79_pre,pdf80_pre,pdf81_pre,pdf82_pre,pdf83_pre,pdf84_pre,pdf85_pre,pdf86_pre,pdf87_pre,pdf88_pre,pdf89_pre,pdf90_pre,pdf91_pre,pdf92_pre,pdf93_pre,pdf94_pre,pdf95_pre,pdf96_pre,pdf97_pre,pdf98_pre,pdf99_pre,pdf100_pre,psweight1_pre,psweight2_pre,psweight3_pre,psweight4_pre,scale1_pre,scale2_pre,scale3_pre,scale4_pre,scale5_pre,scale6_pre,weight,hemfilter,wopu,wol1,woeff,wotop,wobsf,weighthdampUp,weighthdampDown,weightPSUp,weightPSDown,weightJECUp,weightJECDown,weightJERUp,weightJERDown,weightpuweightUp,weightpuweightDown,weightl1prefiringUp,weightl1prefiringDown,weighteffSFUp,weighteffSFDown,weighteffrecoSFUp,weighteffrecoSFDown,weighttopSFUp,weighttopSFDown,weightgenweightUp,weightgenweightDown,weighttrigSFUp,weighttrigSFDown,weightbtagSFbcUp,weightbtagSFbcDown,weightbtagSFbc_correlatedUp,weightbtagSFbc_correlatedDown,weightbtagSFlightUp,weightbtagSFlightDown,weightbtagSFlight_correlatedUp,weightbtagSFlight_correlatedDown
0,233.175064,214.532135,87.888283,63.678883,213.835953,188.569855,3.044904,2.723889,171.521591,0.284261,25.266092,3.538699,2.478114,-1.495361,0.307678,2,2,2,2,2,2,226.606232,1.712373,207.538727,1.546143,-1.846924,1.391357,1.0,1.0,1.0,1.0,0.844028,0.645753,1.126787,0.993064,0.994683,0.991446,1.0,1.0,1.0,0.988460,0.990014,0.986908,0.996823,0.996911,0.996733,0.939449,1.0,0.939449,0.901123,0.935279,0.867419,0.962177,0.841829,0.901123,0.901123,0.901123,0.901123,0.900859,0.901377,0.901127,0.901119,0.999969,0.999268,1.000427,0.999481,1.002808,0.999939,0.997559,0.999664,0.998138,1.001465,0.999451,0.996338,0.998566,0.996216,0.999115,0.999542,1.001404,0.999176,0.998260,1.000488,0.998810,1.000854,0.998047,1.001282,0.997559,0.998810,1.001465,0.997681,0.998352,1.008362,1.001099,1.000549,1.002197,1.001404,1.000183,0.998810,0.998474,1.001099,0.999939,1.000488,0.999634,0.998169,0.996124,1.001526,1.001526,0.999298,1.002380,1.001953,0.998962,0.999786,1.001648,1.000854,0.998718,0.999756,1.000549,1.000732,1.000000,0.999084,1.000183,0.999817,1.000488,0.999756,0.999908,0.999695,0.999451,1.000061,0.999756,1.000061,1.000488,1.000244,0.999603,0.999115,1.000061,0.999817,0.999573,0.999756,1.000244,0.999329,1.000122,1.000305,1.000061,1.000305,1.000366,0.999908,0.999939,0.999817,1.000000,0.999817,1.000122,1.000000,0.999939,0.999939,0.999908,1.000061,1.000000,1.000122,0.999817,1.000366,0.999878,

In [16]:
with open(out_dir+'/Nevents.txt', 'a') as f:
    f.write(eventweights)

# Preselection

In [17]:
dic_df_list = [dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_jer_u, dic_df_met_jer_d, dic_df_met_ue_u, dic_df_met_ue_d, dic_df_scale_u, dic_df_scale_d]
dic_key_list = ['raw', 'met_u', 'met_d', 'met_jer_u', 'met_jer_d', 'met_ue_u', 'met_ue_d', 'scale_u', 'scale_d']

In [18]:
def get_nested_dic(dic_df_list, dic_key_list, query_str):
    weihtedevents = """
    #############################################################
    ####   unweighted (weighted) #events after preselection  ####
    #############################################################
    """

    nested_dic_df={}

    for i, dic in enumerate(dic_df_list):
        dic_tmp = {}
        for key in dic:
#             query_str_new = query_str
#             if "18" in ERA and key=='Data':
#                 query_str_new += " and hemveto==1"
#             if "18" in ERA and key!='Data':
#                 query_str_new += " and hemfilter>0"
            dic_tmp[key]=dic[key].query(query_str)
            unweighted_nevt = dic_tmp[key].shape[0]
            if dic_key_list[i]=='raw':
                print(key, dic_key_list[i], ":", unweighted_nevt, "(", sum(dic_tmp[key]["weight"]), ")")  
                weihtedevents+=key+":"+str(unweighted_nevt)+"("+str(sum(dic_tmp[key]["weight"]))+")\n"
            else:
                print(key, dic_key_list[i], ":", dic_tmp[key].shape[0])
        #     print(key, ":", unweighted_nevt, "(", unweighted_nevt*lumi*dic_xsec[key]/merged_dic_df[key][0], ")")
        nested_dic_df[dic_key_list[i]]=dic_tmp
    
    return nested_dic_df, weihtedevents

In [19]:
# query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"

# nested_dic_df, weihtedevents = get_nested_dic(dic_df_list, dic_key_list, query_str)

In [20]:
# with open(out_dir+'/Nevents.txt', 'a') as f:
#     f.write(weihtedevents)

In [21]:
# if DUMP_PKL or "16" in ERA:
#     print("Dump pickle")
#     with open(out_dir+"/"+"nested_dic_df.pkl", "wb") as f:
#         pickle.dump(nested_dic_df, f)

In [ ]:
def dump_pkl(dic_df_list, dic_key_list, region, base_line, nbcut):
    nested_dic_df, weihtedevents = get_nested_dic(dic_df_list, dic_key_list, base_line+" and nbjetflav_pre"+nbcut)
    # nbjet_jec
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jesup_pre', 'weightJECUp', 'hemfilter']    
    for key in dic_df:  # global scope
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jesup_pre"+nbcut)
        unweighted_nevt = dic_tmp[key].shape[0]
    nested_dic_df['jes_u'] = dic_tmp
    # nbjet_jec
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jesdown_pre', 'weightJECDown', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jesdown_pre"+nbcut)
        unweighted_nevt = dic_tmp[key].shape[0]    
    nested_dic_df['jes_d'] = dic_tmp
    # nbjet_jer
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jerup_pre', 'weightJERUp', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jerup_pre"+nbcut)
        unweighted_nevt = dic_tmp[key].shape[0]
    nested_dic_df['jer_u'] = dic_tmp
    # nbjet_jer
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jerdown_pre', 'weightJERDown', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jerdown_pre"+nbcut)
        unweighted_nevt = dic_tmp[key].shape[0]    
    nested_dic_df['jer_d'] = dic_tmp    
    with open(out_dir+"/"+"nested_dic_df_"+region+".pkl", "wb") as f:
        pickle.dump(nested_dic_df, f)     
    with open(out_dir+"/"+"nested_dic_df_"+region+"_raw.pkl", "wb") as f:
        pickle.dump(nested_dic_df['raw'], f) 
        
    nested_dic_df.clear()

# b tag selection

In [ ]:
if "18" in ERA:
    base_line = "hemfilter>0 and "
else:
    base_line = ""

if REGION == "SR":
    base_line += "mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"
    dump_pkl(dic_df_list, dic_key_list, REGION+"1", base_line, nbcut="==0")
    base_line += "mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"
    dump_pkl(dic_df_list, dic_key_list, "TTCR", base_line, nbcut=">=0")

if REGION == "SB":
    if "em" in ERA: # for ww
        base_line = "mass_pre>100 and pt_lead_pre>45 and pt_lead_pre<80 and pt_trail_pre>40 and PuppiMET_pre>100"    
        dump_pkl(dic_df_list, dic_key_list, "PT"+REGION, base_line, nbcut="==0")
        base_line = "mass_pre>40 and mass_pre<100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"    
        dump_pkl(dic_df_list, dic_key_list, "MASS"+REGION, base_line, nbcut="==0")
    else: # for dy
        base_line = "mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>30 and PuppiMET_pre<100"    
        dump_pkl(dic_df_list, dic_key_list, "MET"+REGION, base_line, nbcut="==0")
        base_line = "mass_pre>80 and mass_pre<100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"    
        dump_pkl(dic_df_list, dic_key_list, "MASS"+REGION, base_line, nbcut="==0")
 

DY1000To1500 raw : 2795 ( 9.60074007511139 )
DY1500To2000 raw : 6192 ( 2.933961736227502 )
DY2000To3000 raw : 11348 ( 1.2071509899160446 )
DY500To700 raw : 722 ( 23.24059208482504 )
DY700To800 raw : 929 ( 7.53553510364145 )
DY800To1000 raw : 1495 ( 9.874832854606211 )
DY raw : 455 ( 366.6778000295162 )
Data raw : 4462 ( 4462.0 )
HWW raw : 1 ( 0.001919121714308858 )
H raw : 0 ( 0 )
TTDown raw : 27331 ( 2565.200268383138 )
TTSemi raw : 198 ( 8.813359366729856 )
TTUp raw : 29675 ( 2641.2614702172577 )
TTZ raw : 4604 ( 3.7027869490120793 )
TT raw : 76446 ( 2631.3231257288717 )
TThdampDown raw : 27446 ( 2557.200512960553 )
TThdampUp raw : 28170 ( 2670.880312680267 )
WWW raw : 30394 ( 2.247930054861172 )
WWZ raw : 9140 ( 6.639108236769971 )
WW raw : 8447 ( 611.7291469480842 )
WZQQ raw : 497 ( 4.212942417012528 )
WZ raw : 5385 ( 98.90206371038221 )
ZZ raw : 31559 ( 21.429218331228185 )
Zp-1700_CH-345 raw : 13691 ( 129.21648422139697 )
Zp-1700_CH-595 raw : 16769 ( 158.3287568341475 )
Zp-1700_C

DY1000To1500 met_ue_u : 2649
DY1500To2000 met_ue_u : 5898
DY2000To3000 met_ue_u : 10979
DY500To700 met_ue_u : 668
DY700To800 met_ue_u : 883
DY800To1000 met_ue_u : 1417
DY met_ue_u : 433
Data met_ue_u : 4463
HWW met_ue_u : 0
H met_ue_u : 0
TTDown met_ue_u : 26687
TTSemi met_ue_u : 198
TTUp met_ue_u : 28938
TTZ met_ue_u : 4518
TT met_ue_u : 74744
TThdampDown met_ue_u : 26773
TThdampUp met_ue_u : 27463
WWW met_ue_u : 29836
WWZ met_ue_u : 8978
WW met_ue_u : 8297
WZQQ met_ue_u : 420
WZ met_ue_u : 5273
ZZ met_ue_u : 31336
Zp-1700_CH-345 met_ue_u : 13648
Zp-1700_CH-595 met_ue_u : 16710
Zp-1700_CH-845 met_ue_u : 18400
Zp-2100_CH-345 met_ue_u : 15949
Zp-2100_CH-595 met_ue_u : 19034
Zp-2100_CH-845 met_ue_u : 20989
Zp-2500_CH-345 met_ue_u : 17715
Zp-2500_CH-595 met_ue_u : 20422
Zp-2500_CH-845 met_ue_u : 22444
Zp-2900_CH-1095 met_ue_u : 24631
Zp-2900_CH-1345 met_ue_u : 25057
Zp-2900_CH-345 met_ue_u : 19319
Zp-2900_CH-595 met_ue_u : 21622
Zp-2900_CH-845 met_ue_u : 23371
Zp-3300_CH-1095 met_ue_u : 2

Zp-3700_CH-595 raw : 809 ( 0.06592837440985022 )
Zp-3700_CH-845 raw : 0 ( 0 )
Zp-4100_CH-1095 raw : 951 ( 0.033979540617110615 )
Zp-4100_CH-1345 raw : 933 ( 0.03302104569047515 )
Zp-4100_CH-1595 raw : 909 ( 0.033061233210901264 )
Zp-4100_CH-1845 raw : 971 ( 0.034567083163892676 )
Zp-4100_CH-345 raw : 777 ( 0.02753182014112099 )
Zp-4100_CH-595 raw : 892 ( 0.03190180038927792 )
Zp-4100_CH-845 raw : 890 ( 0.03175749211823131 )
antitop raw : 1530 ( 359.86114730127156 )
top raw : 1468 ( 360.4382129050791 )
DY1000To1500 met_u : 115
DY1500To2000 met_u : 231
DY2000To3000 met_u : 415
DY500To700 met_u : 52
DY700To800 met_u : 46
DY800To1000 met_u : 75
DY met_u : 95
Data met_u : 9711
HWW met_u : 0
H met_u : 0
TTDown met_u : 101694
TTSemi met_u : 675
TTUp met_u : 110233
TTZ met_u : 18400
TT met_u : 283937
TThdampDown met_u : 102168
TThdampUp met_u : 104999
WWW met_u : 2236
WWZ met_u : 1555
WW met_u : 331
WZQQ met_u : 80
WZ met_u : 228
ZZ met_u : 766
Zp-1700_CH-345 met_u : 440
Zp-1700_CH-595 met_u :

TThdampUp met_ue_d : 101139
WWW met_ue_d : 2187
WWZ met_ue_d : 1515
WW met_ue_d : 329
WZQQ met_ue_d : 70
WZ met_ue_d : 222
ZZ met_ue_d : 763
Zp-1700_CH-345 met_ue_d : 442
Zp-1700_CH-595 met_ue_d : 463
Zp-1700_CH-845 met_ue_d : 565
Zp-2100_CH-345 met_ue_d : 512
Zp-2100_CH-595 met_ue_d : 548
Zp-2100_CH-845 met_ue_d : 596
Zp-2500_CH-345 met_ue_d : 589
Zp-2500_CH-595 met_ue_d : 636
Zp-2500_CH-845 met_ue_d : 691
Zp-2900_CH-1095 met_ue_d : 782
Zp-2900_CH-1345 met_ue_d : 802
Zp-2900_CH-345 met_ue_d : 598
Zp-2900_CH-595 met_ue_d : 696
Zp-2900_CH-845 met_ue_d : 708
Zp-3300_CH-1095 met_ue_d : 831
Zp-3300_CH-1595 met_ue_d : 836
Zp-3300_CH-345 met_ue_d : 646
Zp-3300_CH-595 met_ue_d : 764
Zp-3700_CH-1095 met_ue_d : 928
Zp-3700_CH-1345 met_ue_d : 874
Zp-3700_CH-1595 met_ue_d : 911
Zp-3700_CH-1845 met_ue_d : 859
Zp-3700_CH-345 met_ue_d : 766
Zp-3700_CH-595 met_ue_d : 808
Zp-3700_CH-845 met_ue_d : 881
Zp-4100_CH-1095 met_ue_d : 950
Zp-4100_CH-1345 met_ue_d : 935
Zp-4100_CH-1595 met_ue_d : 910
Zp-4100_

In [23]:
sys_list = ["scale", "puweight", "l1prefiring", "effrecoSF", "effSF", "topSF", "met", "metjer", "metue", "trigSF", "hdamp", "PS", "btagSF", "jes", "jer"]
btag_sys = ['btagSFbc', 'btagSFbc_correlated', 'btagSFlight', 'btagSFlight_correlated']
sys_list_new = sys_list + btag_sys
sys_list_new.remove('btagSF')
print(sys_list_new)

['scale', 'puweight', 'l1prefiring', 'effrecoSF', 'effSF', 'topSF', 'met', 'metjer', 'metue', 'trigSF', 'hdamp', 'PS', 'jes', 'jer', 'btagSFbc', 'btagSFbc_correlated', 'btagSFlight', 'btagSFlight_correlated']
